# Imbalance Strategy Comparison

Compare 6 strategies for handling the 12.1:1 class imbalance in WDR91 DEL data.

All strategies use **ECFP6 fingerprints** as features.
**CV:** StratifiedGroupKFold (5 folds, grouped by DEL library prefix) to prevent library leakage.
**Primary metric:** Enrichment Factor @1% (EF1)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils.data_loader import load_fingerprints
from src.models.imbalance_comparison import run_comparison, summarize

sns.set_theme(style='whitegrid', font_scale=1.1)
FIGURES = Path('../outputs/figures')
RESULTS = Path('../outputs/predictions')

DATA_PATH = '../data/raw/WDR91.parquet'

## 1. Load ECFP6 Fingerprints

In [ ]:
print('Loading ECFP6 fingerprints (sparse)...')
X, df = load_fingerprints(DATA_PATH, fp_name='ECFP6')

y = df['LABEL'].values
groups = df['lib_prefix'].values

print(f'X shape   : {X.shape}  (sparse, nnz={X.nnz:,})')
print(f'y hits    : {y.sum():,} / {len(y):,} ({100*y.mean():.2f}%)')
print(f'Groups    : {len(set(groups))} unique DEL libraries')

## 2. Run 5-Fold CV for All Strategies

In [ ]:
results = run_comparison(X, y, groups, n_splits=5)
results.to_csv(RESULTS / 'imbalance_comparison_cv.csv', index=False)
print('\nRaw results saved.')
results.head(10)

## 3. Summary Table

In [ ]:
summary = summarize(results)
print(summary.to_string())
summary.to_csv(RESULTS / 'imbalance_comparison_summary.csv')
summary

## 4. Visualise Results

In [ ]:
metrics = ['ef1', 'ef5', 'auprc', 'auroc']
titles  = ['Enrichment Factor @1% (higher=better)', 'Enrichment Factor @5%',
           'AUPRC (higher=better)', 'AUROC']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
strategy_order = summary.index.tolist()  # sorted by EF1

for ax, metric, title in zip(axes.flat, metrics, titles):
    means = summary[f'{metric}_mean'][strategy_order]
    stds  = summary[f'{metric}_std'][strategy_order]
    colors = ['coral' if i == 0 else 'steelblue' for i in range(len(strategy_order))]
    bars = ax.bar(strategy_order, means, yerr=stds, capsize=4,
                  color=colors, edgecolor='white', error_kw={'elinewidth':1.5})
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(metric.upper())
    ax.tick_params(axis='x', rotation=30)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + stds[bar.get_x()] * 0.1 if False else bar.get_height()*1.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Imbalance Strategy Comparison — WDR91 DEL (5-fold CV)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'imbalance_comparison.png', dpi=150)
plt.show()
print('Figure saved.')

## 5. Per-Fold Detail (EF1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for strategy in strategy_order:
    fold_data = results[results['strategy'] == strategy]
    ax.plot(fold_data['fold'], fold_data['ef1'], marker='o', label=strategy)
ax.set_xlabel('CV Fold')
ax.set_ylabel('Enrichment Factor @1%')
ax.set_title('EF@1% Per Fold by Strategy')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIGURES / 'ef1_per_fold.png', dpi=150)
plt.show()

## 6. Conclusion

The best strategy from this comparison will be used for final model training on the full dataset, followed by prospective prediction on the 4.4M compound library.